In [ ]:
import torch.nn.functional as F
import os
import sys

sys.path.append("..")
from src.data import NiiPoint, CONSTANTS

from src.architectures import NiiKNN, NiiCKNN
        
test_path_img = os.path.join("../", CONSTANTS.NORM_DATA_PATH, f"imagesVl/")
test_path_label = os.path.join("../", CONSTANTS.NORM_DATA_PATH, f"labelsVl/")

train_path_img = os.path.join("../", CONSTANTS.NORM_DATA_PATH, "imagesTr")
train_path_label = os.path.join("../", CONSTANTS.NORM_DATA_PATH, "labelsTr")

target_path_label_cknn = os.path.join("../nnUNet_raw/Predictions_val/cknn/")
target_path_label_knn = os.path.join("../nnUNet_raw/Predictions_val/knn/")
os.makedirs(target_path_label_cknn, exist_ok=True)
os.makedirs(target_path_label_knn, exist_ok=True)

for file_path in os.listdir(test_path_img):
    file_id = file_path[:9]
    img_path = os.path.join(test_path_img, file_path)
    lab_path = os.path.join(test_path_label, f"{file_id}_0001.nii.gz")
    x = NiiPoint.from_path(img_path, "cuda")
    y = NiiPoint.from_path(lab_path, "cuda")


    model = NiiKNN(3, train_path_img, train_path_label, max_num=100)
    x_pred, y_pred = model(x)
    y_pred.save_to_path(os.path.join(target_path_label_knn, f"{file_id}_0001.nii.gz"))
    print(f"Predicted label for {file_id} saved to {os.path.join(target_path_label_knn, f'{file_id}_0001.nii.gz')}")
    
    fs = 5
    model = NiiCKNN(3, (fs, fs, fs), train_path_img, train_path_label, max_num=4, max_batch=15)
    x_pred, y_pred = model(x)
    y_pred.save_to_path(os.path.join(target_path_label_cknn, f"{file_id}_0001.nii.gz"))
    print(f"Predicted label for {file_id} saved to {os.path.join(target_path_label_cknn, f'{file_id}_0001.nii.gz')}")

In [ ]:
import matplotlib.pyplot as plt
slices = (0.5, 0.5 ,0.5)
thicks_ncct = (1, 1, 1)
thicks_vess = (10, 10, 10) # Maximum HU range for a layer of thickness 10 voxels 

for case_id in [7, 10, 12]:
    img_path = os.path.join(test_path_img, f"case_{str(case_id).zfill(4)}_0000.nii.gz")
    lab_path = os.path.join(test_path_label, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    lab_path_pred = os.path.join(target_path_label_cknn, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    lab_path_pred_knn = os.path.join(target_path_label_knn, f"case_{str(case_id).zfill(4)}_0001.nii.gz")
    
    img_nii = NiiPoint.from_path(img_path, "cuda")
    lab_nii = NiiPoint.from_path(lab_path, "cuda")
    lab_nii_pred = NiiPoint.from_path(lab_path_pred, "cuda")
    lab_nii_pred_knn = NiiPoint.from_path(lab_path_pred_knn, "cuda")

    plt.figure(figsize=(15, 12))
    axes = [
        [plt.subplot(4, 3, i*3 + j + 1) for j in range(3)] for i in range(4)
    ]
    img_nii.plot_slices(slices, thicks_ncct, axes=axes[0])
    lab_nii.plot_slices(slices, thicks_vess, axes=axes[1])
    lab_nii_pred.plot_slices(slices, thicks_vess, axes=axes[2])
    lab_nii_pred_knn.plot_slices(slices, thicks_vess, axes=axes[3])


FileNotFoundError: No such file or no access: '../nnUNet_raw/Predictions_val/cknn/case_0007_0001.nii.gz'